In [6]:
import optuna
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

import gymnasium as gym
from gymnasium import spaces

import gc

import optuna
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from wandb.integration.sb3 import WandbCallback
import wandb
from alibi_detect.od import IForest

# Environnement Forex compatible Gymnasium

In [7]:
class ForexEnv(gym.Env):
    def __init__(self, df):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.max_steps = len(df) - 1
        self.current_step = 0

        # Action : 0 = hold, 1 = buy, 2 = sell
        self.action_space = spaces.Discrete(3)

        # Observation : prix actuel et delta précédent
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(2,), dtype=np.float32)

        self.position = 0  # 0 = neutral, 1 = long, -1 = short
        self.entry_price = 0.0

    def reset(self, **kwargs):
        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        obs = self._next_obs()
        info = {}
        return obs, info

    def _next_obs(self):
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
        obs = np.array([price, delta], dtype=np.float32)

    def step(self, action):
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
        reward = ... # calcul du reward

        # Vérification des valeurs invalides
        if not np.isfinite(price) or not np.isfinite(delta) or not np.isfinite(reward):
            raise ValueError(f"Invalid values detected at step {self.current_step}: price={price}, delta={delta}, reward={reward}")

        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        truncated = False
        obs = np.array([price, delta], dtype=np.float32)
        info = {}
        return obs, reward, terminated, truncated, info


# Chargement des données EUR/USD

In [8]:
df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
df = df.dropna().reset_index(drop=True)

C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54056\1899308409.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed

# Fonction d'optimisation Optuna

In [9]:
def make_env():
    return Monitor(ForexEnv(df))


# === OPTIMISATION DES HYPERPARAMÈTRES ===
def optimize_ppo(trial):
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-3)
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
    ent_coef = trial.suggest_uniform("ent_coef", 0.0, 0.02)

    env = DummyVecEnv([make_env])
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=gamma,
        ent_coef=ent_coef,
        tensorboard_log="./logs/optuna_forex",
        verbose=0,
    )

    model.learn(total_timesteps=100_000)
   # Après l'évaluation
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5, deterministic=True)

    # Sauvegarder le modèle temporairement si tu veux l'examiner plus tard
    tmp_path = f"tmp_models/trial_{trial.number}.zip"
    model.save(tmp_path)

    # stocke uniquement des infos sérialisables
    trial.set_user_attr("model_path", tmp_path)
    trial.set_user_attr("mean_reward", float(mean_reward))

    # cleanup
    model.env.close()
    del model, env
    import gc; gc.collect()

    return mean_reward

# Lancer la recherche Optuna

In [10]:
study = optuna.create_study(direction="maximize")
study.optimize(optimize_ppo, n_trials=20)

best_trial = max(study.get_trials(deepcopy=False), key=lambda t: t.value if t.value is not None else float('-inf'))
best_model_path = best_trial.user_attrs.get("model_path")
from stable_baselines3 import PPO
if best_model_path:
    best_model = PPO.load(best_model_path)
else:
    # re-entraîner à partir des best_params
    best_params = best_trial.params
    env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
    best_model = PPO("MlpPolicy", env, **best_params)
    best_model.learn(total_timesteps=500_000)
print("✅ Best trial:")
print("  Reward:", best_trial.value)
print("  Params:", best_trial.params)

# === ENTRAÎNEMENT FINAL AVEC W&B ===
best_params = best_trial.params
env = DummyVecEnv([make_env])

wandb.init(
    mode="offline",
    project="forex-ppo-hallucination",
    config=best_params,
)

best_model = PPO("MlpPolicy", env, **best_params, verbose=0)
best_model.learn(
    total_timesteps=500_000,
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{wandb.run.name}",
        verbose=2,
    ),
)
best_model.save("models/best_ppo_forex.zip")
wandb.finish()

[I 2025-10-19 15:32:23,629] A new study created in memory with name: no-name-36ee2e4b-47ba-4ec6-99d9-bebc72c56de9
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54056\3369018511.py:7: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-3)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54056\3369018511.py:10: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_54056\3369018511.py:11: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0

ValueError: Expected parameter logits (Tensor of shape (1, 3)) of distribution Categorical(logits: torch.Size([1, 3])) to satisfy the constraint IndependentConstraint(Real(), 1), but found invalid values:
tensor([[nan, nan, nan]])

In [ ]:
best_model.learn(total_timesteps=500_000)  # ou 1_000_000 si nécessaire
best_model.save("models/best_ppo_forex_final.zip")

# FONCTION DE DÉTECTION D’HALLUCINATIONS

In [ ]:
def detect_hallucinations(model, env, n_steps=5000, threshold=0.05):
    obs = env.reset()
    actions = []
    rewards = []

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, info = env.step(action)
        actions.append(action.flatten())
        rewards.append(np.mean(reward))
        if done:
            obs = env.reset()

    actions = np.array(actions)
    rewards = np.array(rewards)

    # === Détection d’anomalies sur les actions ===
    od = IForest(threshold=threshold)
    od.fit(actions)
    preds = od.predict(actions)

    anomaly_ratio = preds["data"]["is_outlier"].mean()
    print(f"⚠️ Anomaly ratio (hallucination risk): {anomaly_ratio:.2%}")

    if anomaly_ratio > 0.1:
        print("🚨 Possible policy hallucination detected! (actions incohérentes)")
    else:
        print("✅ Policy seems stable and consistent.")

    return anomaly_ratio

# ÉVALUATION DU COMPORTEMENT DU MODÈLE

In [ ]:
env_test = DummyVecEnv([lambda: Monitor(ForexEnv(df))])

detect_hallucinations(best_model, env_test, n_steps=5000, threshold=0.05)
env.close()
tmp.oubeur